# QLoRA Fine-tuning — Network Fault Explainer

This notebook fine-tunes a small instruct model with **QLoRA (4-bit quantised Low-Rank Adaptation)**  
so it learns to produce structured fault explanations for network monitoring events.

**What you need:**
- A free Colab T4 GPU (Runtime → Change runtime type → T4 GPU)
- A Hugging Face account + write token (for gated models) — paste it when prompted

**Pipeline at a glance:**
```
train.jsonl  ──┐
               ├─► format with chat template ─► SFTTrainer (LoRA only) ─► adapter/
val.jsonl    ──┘                                         │
                                                         └─► loss_curve.png
```

## 0 · Mount Google Drive and clone the repo

We store the adapter and results on Drive so they survive a runtime disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Clone your repo (or adjust the path if you uploaded it manually) ────────
import os

REPO_URL   = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← change this
REPO_DIR   = '/content/project'
DRIVE_SAVE = '/content/drive/MyDrive/fault-lora-outputs'       # where we'll copy results

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.makedirs(DRIVE_SAVE, exist_ok=True)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## 1 · Install dependencies

We pin exact versions so this cell is reproducible across Colab sessions.  
The `--quiet` flag hides the flood of pip output.

In [ ]:
!pip install --quiet \
    'transformers>=4.40.0' \
    'peft>=0.11.0' \
    'trl>=0.9.0' \
    'bitsandbytes>=0.43.0' \
    'accelerate>=0.29.0' \
    'datasets>=2.18.0' \
    'sentencepiece>=0.2.0' \
    'matplotlib>=3.8.0'

print('Installation complete.')

## 2 · Configuration

All tuneable knobs live here.  Change `BASE_MODEL` to try a different model  
(e.g. `Qwen/Qwen2.5-0.5B-Instruct` for faster experiments).

In [ ]:
from pathlib import Path

# ── Model ───────────────────────────────────────────────────────────────────
BASE_MODEL   = 'Qwen/Qwen2.5-1.5B-Instruct'

# ── LoRA ────────────────────────────────────────────────────────────────────
# r (rank): how many dimensions in the low-rank matrices A and B.
# Higher r → more parameters → better fit but also more VRAM.
LORA_R       = 16
# alpha: scaling factor applied to the LoRA output (ΔW = alpha/r · B·A).
# alpha=2r is a common default that keeps the effective learning rate stable.
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ── Training ─────────────────────────────────────────────────────────────────
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-4
BATCH_SIZE    = 2       # micro-batch; small to fit T4 16 GB
GRAD_ACCUM    = 8       # effective batch = 2 × 8 = 16
MAX_SEQ_LEN   = 512     # truncate longer examples
WARMUP_RATIO  = 0.05

# ── Paths ────────────────────────────────────────────────────────────────────
LLM_DIR    = Path(REPO_DIR) / 'llm'
TRAIN_FILE = LLM_DIR / 'data' / 'train.jsonl'
VAL_FILE   = LLM_DIR / 'data' / 'val.jsonl'
OUT_DIR    = LLM_DIR / 'outputs' / 'adapter'
RES_DIR    = LLM_DIR / 'results'

OUT_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Model  : {BASE_MODEL}')
print(f'LoRA   : r={LORA_R}, alpha={LORA_ALPHA}')
print(f'Train  : {sum(1 for _ in open(TRAIN_FILE))} examples')
print(f'Val    : {sum(1 for _ in open(VAL_FILE))} examples')

## 3 · Check GPU

Verify that Colab assigned a T4 (or better).  
If you see `No GPU`, go to Runtime → Change runtime type → GPU.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}  ({vram_gb:.1f} GB VRAM)')
    print(f'BF16 supported: {torch.cuda.is_bf16_supported()}')
else:
    print('No GPU — training will be very slow.  Enable GPU in Runtime settings.')

## 4 · Load tokeniser and format the dataset

Each example has three fields: `instruction`, `input`, `output`.  
We combine them into a full conversation using the model's own **chat template**.

The chat template adds the special tokens the model was pre-trained with  
(e.g. `<|im_start|>user` … `<|im_end|>` for Qwen2.5).  
Using the correct template is critical — a mismatch degrades generation quality.

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Many causal LMs ship without a pad token; reuse eos_token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # right-pad for causal LM training

print('Vocab size :', tokenizer.vocab_size)
print('EOS / PAD  :', tokenizer.eos_token, '/', tokenizer.pad_token)

In [ ]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

def format_example(ex):
    """Merge instruction + input into a user turn; output is the assistant turn.

    apply_chat_template returns a plain string with the model's special tokens
    already embedded.  SFTTrainer will tokenise this string during training.
    """
    messages = [
        {'role': 'user',      'content': f"{ex['instruction']}\n\n{ex['input']}"},
        {'role': 'assistant', 'content': ex['output']},
    ]
    return {'text': tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_ds = Dataset.from_list([format_example(r) for r in load_jsonl(TRAIN_FILE)])
val_ds   = Dataset.from_list([format_example(r) for r in load_jsonl(VAL_FILE)])

print(f'Train : {len(train_ds)} examples')
print(f'Val   : {len(val_ds)} examples')
print('\n── First training example (truncated) ──')
print(train_ds[0]['text'][:600], '…')

## 5 · Load model in 4-bit (QLoRA)

**BitsAndBytes NF4 quantisation** compresses each 16-bit weight to 4 bits,  
reducing Qwen2.5-1.5B from ~3 GB to ~900 MB — easily fitting on a T4.

The model weights are **frozen** at this stage; only the LoRA matrices will train.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # Normal Float 4: best for normally distributed weights
    bnb_4bit_use_double_quant=True,     # quantise the quant constants too → ~0.4 fewer bits/param
    bnb_4bit_compute_dtype=torch.bfloat16,  # up-cast to bf16 for actual matrix multiplications
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',          # automatically distributes layers across GPU (and CPU if needed)
    trust_remote_code=True,
)

# Disable KV-cache during training — incompatible with gradient checkpointing
model.config.use_cache = False

print('Model loaded.  Memory allocated:', round(torch.cuda.memory_allocated() / 1e9, 2), 'GB')

## 6 · Apply LoRA adapter

`prepare_model_for_kbit_training` does two things before PEFT wraps the model:
1. Casts layer norm weights to float32 (more numerically stable).
2. Enables gradient checkpointing (trade compute for memory).

`get_peft_model` then injects the LoRA A·B matrices beside every linear layer  
and freezes all original weights.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# Prepare base model for 4-bit training (cast norms to fp32, enable grad checkpointing)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,               # rank of the adapter matrices (A: d×r, B: r×d)
    lora_alpha=LORA_ALPHA,  # output is scaled by alpha/r before adding to original weight
    lora_dropout=LORA_DROPOUT,
    target_modules='all-linear',  # apply to all nn.Linear layers (q,k,v,o + FFN projections)
    bias='none',            # don't adapt bias vectors
)

model = get_peft_model(model, lora_config)

# This prints something like: trainable params: 6,815,744 || all params: 1,550,000,000
# Only ~0.4% of weights are actually trained — that's the power of LoRA.
model.print_trainable_parameters()

## 7 · Training arguments

`SFTConfig` extends `TrainingArguments` with SFT-specific knobs  
(`max_seq_length`, `dataset_text_field`, `packing`).  
SFTTrainer automatically masks the prompt tokens in the loss calculation  
so we only train on the **assistant** response.

In [ ]:
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=str(OUT_DIR),

    # ── Schedule ────────────────────────────────────────────────────────────
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',   # LR follows a cosine curve from peak → ~0 over training

    # ── Memory ──────────────────────────────────────────────────────────────
    gradient_checkpointing=True,  # recompute activations on backward to save VRAM
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=False,

    # ── Evaluation & checkpointing ──────────────────────────────────────────
    eval_strategy='epoch',        # compute val loss after every epoch
    save_strategy='epoch',
    load_best_model_at_end=True,  # restore the checkpoint with the lowest val loss
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # ── Logging ─────────────────────────────────────────────────────────────
    logging_steps=10,
    report_to='none',             # disable wandb/tensorboard

    # ── SFT-specific ─────────────────────────────────────────────────────────
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',    # the column our format_example() produced
    packing=False,                # don't concatenate examples — keeps each independently masked
)

print('Training args ready.')
print(f'  Effective batch size : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Max sequence length  : {MAX_SEQ_LEN} tokens')

## 8 · Loss logger callback

The Hugging Face Trainer exposes a callback API.  We use it to collect  
every logged loss value so we can plot a per-epoch loss curve after training.

In [ ]:
class LossLogger(TrainerCallback):
    """Collect training and validation loss from the Trainer's log dictionary."""

    def __init__(self):
        self.logs = []  # list of raw log dicts

    def on_log(self, args, state, control, logs=None, **kwargs):
        # The Trainer calls this after every logging step and after each eval
        if logs:
            self.logs.append(dict(logs))

    def get_epoch_losses(self):
        """Average per-epoch train and val losses from raw logs."""
        train_by_epoch, eval_by_epoch = {}, {}
        for log in self.logs:
            epoch = int(log.get('epoch', 0))
            if 'loss' in log:
                train_by_epoch.setdefault(epoch, []).append(log['loss'])
            if 'eval_loss' in log:
                eval_by_epoch.setdefault(epoch, []).append(log['eval_loss'])

        epochs = sorted(set(list(train_by_epoch) + list(eval_by_epoch)))
        train_losses = [sum(train_by_epoch[e]) / len(train_by_epoch[e])
                        for e in epochs if e in train_by_epoch]
        val_losses   = [sum(eval_by_epoch[e])  / len(eval_by_epoch[e])
                        for e in epochs if e in eval_by_epoch]
        return train_losses, val_losses

loss_logger = LossLogger()
print('Callback instantiated.')

## 9 · Train

This cell will take **~30–50 minutes** on a T4 for 3 epochs over 664 examples.  
The Trainer prints a progress bar and logs loss every 10 steps.

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    callbacks=[loss_logger],
)

print('Starting training …')
trainer.train()
print('Training complete.')

## 10 · Save the LoRA adapter

We save **only the adapter** — a few MB of weight deltas — not the 3 GB base model.  
To run inference later:
```python
from peft import PeftModel
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, ...)
model = PeftModel.from_pretrained(model, 'outputs/adapter')
# or bake in permanently:
model = model.merge_and_unload()
```

In [ ]:
trainer.model.save_pretrained(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f'Adapter saved to {OUT_DIR}')

# List what was saved
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:<40} {size_kb:>6} KB')

## 11 · Loss curve

A well-trained LoRA should show **both losses decreasing and staying close together**.  
A large gap between train and val loss indicates overfitting.

In [ ]:
import matplotlib.pyplot as plt

train_losses, val_losses = loss_logger.get_epoch_losses()
print(f'Train losses per epoch : {[round(l, 4) for l in train_losses]}')
print(f'Val   losses per epoch : {[round(l, 4) for l in val_losses]}')

epochs = list(range(1, max(len(train_losses), len(val_losses)) + 1))

fig, ax = plt.subplots(figsize=(7, 4))
if train_losses:
    ax.plot(epochs[:len(train_losses)], train_losses,
            marker='o', label='Train loss', color='#2563eb')
if val_losses:
    ax.plot(epochs[:len(val_losses)],   val_losses,
            marker='s', label='Val loss',   color='#dc2626', linestyle='--')

ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-entropy loss')
ax.set_title(f'QLoRA fine-tuning — {BASE_MODEL.split("/")[-1]}')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
fig.tight_layout()

curve_path = RES_DIR / 'loss_curve.png'
fig.savefig(curve_path, dpi=150)
plt.show()
print(f'Saved to {curve_path}')

## 12 · Copy outputs to Google Drive

In [ ]:
import shutil

drive_adapter = Path(DRIVE_SAVE) / 'adapter'
drive_results = Path(DRIVE_SAVE) / 'results'

shutil.copytree(OUT_DIR, drive_adapter, dirs_exist_ok=True)
shutil.copytree(RES_DIR, drive_results, dirs_exist_ok=True)

print(f'Adapter  → {drive_adapter}')
print(f'Results  → {drive_results}')

## 13 · Quick inference test

Sanity-check: feed the fine-tuned model a fault event and inspect the output.  
We use `greedy` decoding (temperature=0) for a deterministic, clean result.

In [ ]:
INSTRUCTION = (
    'You are a network operations assistant. '
    'Given a fault event detected on a network device, produce a structured explanation '
    'with three clearly labelled sections: '
    "'Explanation' (what is happening and why it matters, referencing the specific metrics), "
    "'Likely causes' (ranked list of probable root causes for this device type and fault pattern), "
    "and 'Recommended actions' (concrete remediation steps in priority order)."
)

TEST_INPUT = """Device: core-sw-01 (switch)
Fault type: HIGH_CPU_USAGE
Severity: critical
CPU usage: 94%
Memory usage: 71%
Fault duration: 25 minutes"""

messages = [
    {'role': 'user', 'content': f'{INSTRUCTION}\n\n{TEST_INPUT}'}
]

# Tokenise and move to GPU
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,   # adds the assistant-turn opening token
    return_tensors='pt',
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=400,
        do_sample=False,     # greedy decoding
        temperature=1.0,     # irrelevant when do_sample=False but required by some versions
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens (skip the prompt)
generated = tokenizer.decode(
    output_ids[0][input_ids.shape[-1]:],
    skip_special_tokens=True,
)

print('── Model output ──────────────────────────────────────────')
print(generated)